In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
act1b_reference.py
compute the ACT1B in-track reference over the full arc

Replaces the Nov-1-only CFG.ACT1B_FT_REFERENCE (-1.4073e-07) with a value
computed over the same Nov 1-14 span as the GPS arc.


THE BIAS CAVEAT - READ BEFORE USING THE NUMBER!

ACT1B is a "Transformed" product (accelerometer frame -> SRF). It is NOT
bias-calibrated. Bias and scale factors are estimated downstream in POD.
Proof: on Nov 1, lin_accl_y has mean -1.04e-05 with std 1.9e-08, a
near-constant offset ~100x larger than any real nongravitational force on
GRACE-FO (drag ~1e-7, SRP ~1e-8). That's instrument bias.

So for every axis:

    daily_mean = (true nongravitational acceleration) + (instrument bias)

The ABSOLUTE level therefore can't serve as a ground-truth magnitude.

BUT accelerometer bias drifts only slowly, so over a 14-day span it's nearly
constant. That means the DIFFERENCES between daily means are essentially bias-free:

    daily_mean[d] - daily_mean[Nov 1]  ~  drag[d] - drag[Nov 1]

The "delta vs day 1" column below is therefore a genuine, bias-cancelled
measurement of how the drag changed across the arc. This tests the question:
Is the iPINN's 14-day in-track estimate (~-2.9e-07) larger
than the Nov-1 ACT1B value (-1.29e-07) because the drag really rose over
the two weeks, or because the estimate is biased?

  * If the daily means trend strongly more negative, then drag really did rise,
    and the 14-day iPINN value is plausible.
  * If the daily means are flat near -1.3e-07, then the arc-mean drag did NOT
    change, and the iPINN's larger magnitude needs a different explanation.
"""

import os
from glob import glob

import numpy as np
import matplotlib.pyplot as plt


class CFG:
    DATA_FOLDER   = "data"
    ACT1B_PATTERN = "ACT1B_2025-11-*_C_04.txt"
    ORBIT_SEC     = 5624.0          # GRACE-FO orbital period


def parse_act1b(path):
    """ACT1B ASCII -> (t_gps, ax, ay, az, qualflg). Columns after the YAML
    header: gps_time, id, lin_accl_x/y/z, ang_accl_x/y/z, acl_*_res, qualflg."""
    t, ax, ay, az, q = [], [], [], [], []
    in_header = True
    with open(path) as fh:
        for line in fh:
            if in_header:
                if '# End of YAML header' in line:
                    in_header = False
                continue
            p = line.split()
            if len(p) < 12 or p[1] != 'C':
                continue
            try:
                t.append(float(p[0]))
                ax.append(float(p[2])); ay.append(float(p[3])); az.append(float(p[4]))
                q.append(int(p[11], 2))
            except ValueError:
                continue
    return (np.array(t), np.array(ax), np.array(ay), np.array(az),
            np.array(q, dtype=int))


def day_label(path):
    """'.../ACT1B_2025-11-03_C_04.txt' -> '2025-11-03'."""
    base = os.path.basename(path)
    parts = base.split('_')
    return parts[1] if len(parts) > 1 else base


def main():
    files = sorted(glob(os.path.join(CFG.DATA_FOLDER, CFG.ACT1B_PATTERN)))
    if not files:
        raise FileNotFoundError(
            f"No ACT1B files: {os.path.join(CFG.DATA_FOLDER, CFG.ACT1B_PATTERN)}")
    print(f"Found {len(files)} ACT1B file(s)\n")

    print("=" * 88)
    print("PER-DAY STATISTICS  (lin_accl_x = along-track; all values m/s^2)")
    print("=" * 88)
    print(f"  {'date':>12} {'n_clean':>8} {'mean_x':>12} {'median_x':>12} "
          f"{'std_x':>11} {'delta_vs_d1':>12} {'mean_z':>12}")
    print("  " + "-" * 84)

    days, means_x, medians_x, stds_x, means_y, means_z, counts = [], [], [], [], [], [], []
    all_t, all_x = [], []

    for f in files:
        t, ax, ay, az, q = parse_act1b(f)
        good = q == 0
        if good.sum() == 0:
            print(f"  {day_label(f):>12}  (no clean records -- skipped)")
            continue
        t, ax, ay, az = t[good], ax[good], ay[good], az[good]

        days.append(day_label(f))
        counts.append(int(good.sum()))
        means_x.append(ax.mean());  medians_x.append(np.median(ax))
        stds_x.append(ax.std())
        means_y.append(ay.mean());  means_z.append(az.mean())
        all_t.append(t); all_x.append(ax)

        delta = means_x[-1] - means_x[0]
        print(f"  {days[-1]:>12} {counts[-1]:>8d} {means_x[-1]:>12.4e} "
              f"{medians_x[-1]:>12.4e} {stds_x[-1]:>11.3e} {delta:>+12.3e} "
              f"{means_z[-1]:>12.4e}")

    means_x  = np.array(means_x);  medians_x = np.array(medians_x)
    stds_x   = np.array(stds_x);   means_y = np.array(means_y)
    counts   = np.array(counts)
    t_all    = np.concatenate(all_t);  x_all = np.concatenate(all_x)

    # ---- arc aggregate ---------------------------------------------------
    arc_mean   = float(np.average(means_x, weights=counts))   # count-weighted
    arc_median = float(np.median(x_all))
    day1_mean  = float(means_x[0])

    print("\n" + "=" * 88)
    print(f"ARC AGGREGATE  ({days[0]} .. {days[-1]}, {len(days)} days, "
          f"{len(x_all)} clean 1-Hz samples)")
    print("=" * 88)
    print(f"  mean   lin_accl_x over arc : {arc_mean:+.4e} m/s^2   <-- new reference")
    print(f"  median lin_accl_x over arc : {arc_median:+.4e} m/s^2")
    print(f"  day-1 (Nov 1) mean         : {day1_mean:+.4e} m/s^2   (old reference basis)")
    print(f"  old CFG value in the iPINN : -1.4073e-07 m/s^2")
    print(f"  arc mean / day-1 mean      : {arc_mean/day1_mean:.4f}")

    #  the bias-cancelled result
    drift      = float(means_x[-1] - means_x[0])
    spread     = float(means_x.max() - means_x.min())
    std_of_day = float(means_x.std())
    print("\n" + "=" * 88)
    print("BIAS-CANCELLED DRAG VARIATION  (differences of daily means)")
    print("=" * 88)
    print(f"  last day - first day : {drift:+.3e} m/s^2")
    print(f"  max - min across days: {spread:.3e} m/s^2")
    print(f"  std of daily means   : {std_of_day:.3e} m/s^2")
    print(f"  mean within-day std  : {stds_x.mean():.3e} m/s^2  "
          f"(orbital modulation -- real signal)")
    print(f"  bias stability check : mean_y ranges "
          f"{means_y.min():.4e} .. {means_y.max():.4e}  "
          f"(drift {means_y.max()-means_y.min():+.2e})")

    # SRF-X sign, and what it implies
    # VERIFIED from formation geometry using GNV1B on 2025-11-08:
    #     (r_D - r_C) . v_hat_C = -198.3 km at 100% of 86400 epochs
    # => C LEADS, so SRF-X on C (pointing C->D) is ANTI-velocity.
    # Drag opposes velocity, so it enters lin_accl_x with a POSITIVE sign:
    #
    #     measured_x = D + bias        with D >= 0 the drag MAGNITUDE
    #
    # Two consequences, both opposite to what an along-velocity X would give:
    #   * bias <= min(measured_x)   [not >= max]. The measured minimum occurs
    #     where drag is weakest, so min(measured) is the tightest upper bound
    #     on the bias, and D = measured - bias is a ** lower ** bound on drag.
    #   * a POSITIVE daily mean means MORE drag, not a thruster event. The
    #     earlier "positive mean => maneuver" screen was inverted and was
    #     discarding genuine high-drag (storm) days,  it has been removed.
    #     THR1B remains the main source for real maneuver epochs.
    day_min = []
    for f in files:
        t, ax, ay, az, q = parse_act1b(f)
        g = q == 0
        if g.sum():
            day_min.append(ax[g].min())
    bias_ub = float(np.min(day_min))          # bias <= this
    D_arc   = arc_mean - bias_ub              # lower bound on arc-mean drag
    D_day1  = day1_mean - bias_ub

    print("\n" + "=" * 88)
    print("SIGN-CORRECTED DRAG ESTIMATE  (SRF-X is ANTI-velocity: C leads)")
    print("=" * 88)
    print(f"  bias upper bound  = min over arc of lin_accl_x = {bias_ub:+.4e} m/s^2")
    print(f"  arc-mean drag     D >= {D_arc:+.4e} m/s^2   (magnitude, drag opposes v)")
    print(f"  day-1 drag        D >= {D_day1:+.4e} m/s^2")
    print( "  These are LOWER bounds: the true bias may be more negative still,")
    print( "  which would make the true drag larger.")

    print("\n" + "=" * 88)
    print("PASTE INTO THE iPINN CONFIG")
    print("=" * 88)
    print( "  The iPINN reports in-track force in the RTN-T (along-velocity)")
    print( "  convention, where drag is NEGATIVE. So negate the drag magnitude:")
    print(f"    ACT1B_FT_REFERENCE = {-D_arc:.4e}   # m/s^2, RTN-T convention")
    print(f"                                        # arc {days[0]}..{days[-1]}")
    print( "  This is a LOWER BOUND on |drag|; an iPINN magnitude LARGER than")
    print( "  this is consistent, not discrepant. Do not treat the ratio to this")
    print( "  number as an accuracy metric -- use the correlation test instead.")
    print("=" * 88)

    # plots
    plt.style.use('default')
    C_A, C_M = '#D55E00', '#0072B2'          # Okabe-Ito vermillion / blue
    fig, ax = plt.subplots(2, 1, figsize=(9, 6.5))
    fig.patch.set_facecolor('white')

    idx = np.arange(len(days))
    ax[0].errorbar(idx, np.array(means_x) * 1e9, yerr=stds_x * 1e9,
                   fmt='o-', color=C_A, ecolor='0.5', capsize=3, lw=1.3, ms=4,
                   label='daily mean $\\pm$ within-day std')
    ax[0].axhline(arc_mean * 1e9, color=C_M, ls='--', lw=1.2,
                  label=f'arc mean {arc_mean:.2e}')
    ax[0].axhline(bias_ub * 1e9, color='k', ls=':', lw=1.2,
                  label=f'bias upper bound {bias_ub:.2e}')
    ax[0].set_xticks(idx)
    ax[0].set_xticklabels([d[5:] for d in days], rotation=45, fontsize=7.5)
    ax[0].set_ylabel('lin\\_accl\\_x  (nm s$^{-2}$)')
    ax[0].set_title('ACT1B along-track daily means  (SRF-X is anti-velocity: '
                    'more positive = more drag)', fontsize=10)
    ax[0].legend(fontsize=7.5, framealpha=1.0, edgecolor='0.7')
    ax[0].grid(alpha=0.3, lw=0.5)

    # decimate the 1-Hz series to ~1 point per 1/8 orbit for a readable plot
    step = max(1, int(CFG.ORBIT_SEC / 8))
    t_d = (t_all[::step] - t_all[0]) / 86400.0
    ax[1].plot(t_d, x_all[::step] * 1e9, color=C_M, lw=0.5, alpha=0.85)
    ax[1].axhline(arc_mean * 1e9, color=C_A, ls='--', lw=1.2, label='arc mean')
    ax[1].set_xlabel('Days since 2025-11-01')
    ax[1].set_ylabel('lin\\_accl\\_x  (nm s$^{-2}$)')
    ax[1].set_title('ACT1B along-track, full arc (scatter = per-orbit '
                    'drag modulation)', fontsize=10)
    ax[1].legend(fontsize=7.5, framealpha=1.0, edgecolor='0.7')
    ax[1].grid(alpha=0.3, lw=0.5)

    for a in ax:
        a.tick_params(labelsize=8)
        for s in a.spines.values():
            s.set_linewidth(0.8)
    fig.tight_layout()
    out = os.path.join(CFG.DATA_FOLDER, 'act1b_reference.png')
    fig.savefig(out, dpi=300, facecolor='white', bbox_inches='tight')
    print(f"\nSaved plot: {out}")

    np.savez(os.path.join(CFG.DATA_FOLDER, 'act1b_daily_stats.npz'),
             days=np.array(days), mean_x=means_x, median_x=medians_x,
             std_x=stds_x, mean_y=means_y, mean_z=np.array(means_z),
             counts=counts, arc_mean=arc_mean, arc_median=arc_median)
    print("Saved: act1b_daily_stats.npz")


if __name__ == '__main__':
    main()